## human specific genes blast comparison

### upload human specific genes

In [ ]:
import pandas as pd

In [2]:
hsg1=pd.read_excel("/scratch200/reutj/data/human_specific_genes_big.xlsx",sheet_name="HumanSpecific Genes")

/scratch200/reutj/conda-envs/jupyter-scanpy/lib/python3.12/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


In [6]:
hsg1

,Gene Name,Ensembl ID,Chromosome,Gene Type,General Gene Type,Mechanism of Origin,General Mechanism of Origin,Reference(s)
0,39326,ENSG00000122545,7,protein_coding,protein-coding,human-specific gene duplication,gene amplification,9
1,AATBC,ENSG00000215458,21,antisense,long non-coding,hominoid-specific de novo originated protein-c...,de novo origin,1
2,ABCB10,ENSG00000135776,1,protein_coding,protein-coding,human-specific gene amplification,gene amplification,"2,3"
3,ABCC12,ENSG00000140798,16,protein_coding,protein-coding,modern human-specific coding change,gene sequence alteration,4
4,ABCC6,ENSG00000091262,16,protein_coding,protein-coding,human-specific gene duplication or expansion,gene amplification,5
...,...,...,...,...,...,...,...,...
868,ZNF790,ENSG00000197863,19,protein_coding,protein-coding,human-specific gene duplication,gene amplification,22
869,ZNF843,ENSG00000176723,16,protein_coding,protein-coding,human-specific de novo originated protein-codi...,de novo origin,8
870,ZNF85,ENSG00000105750,19,protein_coding,protein-coding,human-specific gene,human-specific gene (undefined feature),29
871,ZNF850,ENSG00000267041,19,protein_coding,protein-coding,human-specific gene loss or pseudogene,gene loss,36


In [3]:
hsgenes2=["SRGAP2","SRGAP2B","SRGAP2C","HYDIN","HYDIN2","ARHGAP11A","ARHGAP11B","FRMPD2","FRMPD2B"
            ,"SRGAP2D","FAM72A","FAM72B","FAM72C","FAM72D","ROCK1","ROCK1P1","PTPN20","PTPN20CP","GTF2I","GTF2IP1","GTF2IP4"
            ,"GTF2IRD2","GTF2IRD2B","GTF2IRD2P1" ,"NCF1","NCF1C","NCF1B","GPR89A","GPR89B","CD8B","CD8B2"
            ,"NOTCH2","NOTCH2NL","NOTCH2NLB","NOTCH2NLC","NOTCH2NLD","CORO1A","CORO1AP","SLX1A","SLX1B"
            ,"BOLA2B","BOLA2","FCGR1A","FCGR1B","FCGR1CP","ARHGEF34P","ARHGEF35","ARHGEF5","CHRFAM7A","CHRNA7"]

In [4]:
import requests

def get_ensembl_gene_id(gene_symbol):
    """
    Retrieve the Ensembl gene ID based on the gene symbol.
    """
    server = "https://rest.ensembl.org"
    endpoint = f"/xrefs/symbol/homo_sapiens/{gene_symbol}?"
    
    response = requests.get(server + endpoint, headers={"Content-Type": "application/json"})
    if not response.ok:
        response.raise_for_status()
    
    data = response.json()
    # Return the first Ensembl ID found
    return data[0]["id"] if data else None

In [5]:
hsg2={}
for gene in hsgenes2:
    hsg2[gene]=get_ensembl_gene_id(gene)
hsg2

{'SRGAP2': 'ENSG00000266028',
 'SRGAP2B': 'ENSG00000196369',
 'SRGAP2C': 'ENSG00000171943',
 'HYDIN': 'ENSG00000157423',
 'HYDIN2': 'ENSG00000276975',
 'ARHGAP11A': 'ENSG00000198826',
 'ARHGAP11B': 'ENSG00000285077',
 'FRMPD2': 'ENSG00000170324',
 'FRMPD2B': 'ENSG00000150175',
 'SRGAP2D': 'ENSG00000270872',
 'FAM72A': 'ENSG00000196550',
 'FAM72B': 'ENSG00000188610',
 'FAM72C': 'ENSG00000263513',
 'FAM72D': 'ENSG00000215784',
 'ROCK1': 'ENSG00000067900',
 'ROCK1P1': 'ENSG00000290877',
 'PTPN20': 'ENSG00000204179',
 'PTPN20CP': 'ENSG00000278561',
 'GTF2I': 'ENSG00000293241',
 'GTF2IP1': 'ENSG00000277053',
 'GTF2IP4': 'ENSG00000233369',
 'GTF2IRD2': 'ENSG00000196275',
 'GTF2IRD2B': 'ENSG00000174428',
 'GTF2IRD2P1': 'ENSG00000214544',
 'NCF1': 'ENSG00000158517',
 'NCF1C': 'ENSG00000165178',
 'NCF1B': 'ENSG00000182487',
 'GPR89A': 'ENSG00000117262',
 'GPR89B': 'ENSG00000117262',
 'CD8B': 'ENSG00000172116',
 'CD8B2': 'ENSG00000254126',
 'NOTCH2': 'ENSG00000134250',
 'NOTCH2NL': 'ENSG00000264

In [6]:
hsg2_df=pd.DataFrame({'gene':hsg2.keys(),'ens':hsg2.values()})

In [7]:
hsg1_df=hsg1.iloc[:,0:2]
hsg1_df=hsg1_df.rename(columns={"Gene Name": "gene", "Ensembl ID": "ens"})

In [8]:
hsg_df=pd.concat([hsg1_df,hsg2_df])
hsg_df=hsg_df.drop_duplicates()

In [25]:
hsg_df.to_csv("/scratch200/reutj/data/hsg_genes.csv")

In [9]:
hsg_ens=hsg_df["ens"]
hsg_symbols=hsg_df["gene"]

In [ ]:
#try with gene_ens..

In [6]:
#get fasta file of 400 last bases from the gene symbols
import requests

def fetch_canonical_transcript_debug(gene_symbol):
    """
    Fetches the last 400 bases of the canonical transcript for a gene, given its symbol.
    Args:
        gene_symbol: Gene symbol (e.g., BRCA2).
    Returns:
        Canonical transcript sequence in FASTA format (last 400 bases).
    """
    server = "https://rest.ensembl.org"
    headers = {"Content-Type": "application/json"}

    # Step 1: Look up the gene by its symbol
    lookup_symbol_endpoint = f"/lookup/symbol/homo_sapiens/{gene_symbol}?expand=1"
    try:
        response = requests.get(server + lookup_symbol_endpoint, headers=headers)
        print(f"Requesting URL: {server + lookup_symbol_endpoint}")
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return None

    if not response.ok:
        print(f"Error during symbol lookup: HTTP {response.status_code} - {response.reason}")
        print(f"Response content: {response.text}")
        return None

    gene_info = response.json()
    #print(f"Gene Info: {gene_info}")

    # Ensure a canonical transcript exists
    canonical_transcript_id = gene_info.get("canonical_transcript")
    if not canonical_transcript_id:
        raise ValueError(f"No canonical transcript found for gene symbol {gene_symbol}")

    # Strip version number if present (e.g., '.1')
    canonical_transcript_id = canonical_transcript_id.split('.')[0]

    # Step 2: Fetch the sequence for the canonical transcript
    seq_endpoint = f"/sequence/id/{canonical_transcript_id}"
    try:
        seq_response = requests.get(server + seq_endpoint, headers=headers)
        print(f"Requesting URL: {server + seq_endpoint}")
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return None

    if not seq_response.ok:
        print(f"Error during sequence fetch: HTTP {seq_response.status_code} - {seq_response.reason}")
        print(f"Response content: {seq_response.text}")
        return None

    sequence = seq_response.json().get("seq")
    if not sequence:
        raise ValueError(f"No sequence found for canonical transcript {canonical_transcript_id}")

    # Get the last 400 bases of the sequence
    last_400_bases = sequence[-400:]

    # Format the result in FASTA format
    fasta_format = f">{canonical_transcript_id} | Last 400 bases\n{last_400_bases}"
    return fasta_format


In [ ]:
#try from the new hsg list

In [2]:
hsg=pd.read_csv("/scratch200/reutj/data/hsg_gene_lists/updated_human_specific_genes2.csv")

In [5]:
hsg_symbols=hsg["gene"].tolist()

In [7]:
hsg_3utr_seq={}
for gene in hsg_symbols:
    hsg_3utr_seq[gene]=fetch_canonical_transcript_debug(gene)

Requesting URL: https://rest.ensembl.org/lookup/symbol/homo_sapiens/SEPTIN7?expand=1
Requesting URL: https://rest.ensembl.org/sequence/id/ENST00000350320
Requesting URL: https://rest.ensembl.org/lookup/symbol/homo_sapiens/AATBC?expand=1
Requesting URL: https://rest.ensembl.org/sequence/id/ENST00000400385
Requesting URL: https://rest.ensembl.org/lookup/symbol/homo_sapiens/ABCB10?expand=1
Requesting URL: https://rest.ensembl.org/sequence/id/ENST00000344517
Requesting URL: https://rest.ensembl.org/lookup/symbol/homo_sapiens/ABCC12?expand=1
Requesting URL: https://rest.ensembl.org/sequence/id/ENST00000311303
Requesting URL: https://rest.ensembl.org/lookup/symbol/homo_sapiens/ABCC6?expand=1
Requesting URL: https://rest.ensembl.org/sequence/id/ENST00000205557
Requesting URL: https://rest.ensembl.org/lookup/symbol/homo_sapiens/ABCD4?expand=1
Requesting URL: https://rest.ensembl.org/sequence/id/ENST00000356924
Requesting URL: https://rest.ensembl.org/lookup/symbol/homo_sapiens/ABHD17A?expand=1

In [8]:
hsg_3utr_seq["NOTCH2NLB"]

'>ENST00000593495 | Last 400 bases\nATTCCAGTTTCTCCACATCCTCATCAACAGTTGTTATTGTCTGTCTTTTTTATTATATTCATCTGTAATGTGAAGTGTTTATCTCATTGTGGTTTTGATTTACATTTCCCTGATGGTTGATGATTTTCAACATCTTTTCATATACTTATTAGTCATTATGTATCTTCTTTGGAGAATGTCTGTTCAGATCCTTTACCTACTTTATAATTGGTTTATCTTTTTAATATTGAACTGTAATAGTTTTTAAAAAATATATCCTAAATACAAGTCTCTTATCAGATAATATGATTTGCAGATATTTTCTGTCATTCTATGTACTGTCTTTTCACATTCTTGATGATAGACTTTTCAGCCCAAATGTTTTTAACTTGATGGAATACAATTTATTTTTTCTTTTGTT'

In [9]:
#clean genes with no seq
no_seq_gene=[]
for i in hsg_3utr_seq.keys():
    if not hsg_3utr_seq[i]:
        no_seq_gene.append(i)
for i in no_seq_gene:
    del hsg_3utr_seq[i]

In [10]:
len(hsg_3utr_seq)

490

In [11]:
hsg_3utr_seq_df=pd.DataFrame({'gene':hsg_3utr_seq.keys(),'seq':hsg_3utr_seq.values()})
hsg_3utr_seq_df.to_csv("/scratch200/reutj/data/hsg_3utr_seq_df2.csv")

In [12]:
#collect all to one fasta file
def save_sequences_to_fasta(sequence_dict, output_file):
    """
    Save a dictionary of sequences to a single FASTA file.
    Args:
        sequence_dict: Dictionary with gene symbols as keys and sequences as values.
        output_file: Path to the output FASTA file.
    """
    with open(output_file, "w") as fasta_file:
        for gene, sequence in sequence_dict.items():
            print(gene, len(sequence))
            # Write in FASTA format
            fasta_file.write(f">{gene}\n")
            # Wrap the sequence at 80 characters for readability
            for i in range(0, len(sequence), 80):
                fasta_file.write(sequence[i:i+80] + "\n")

In [13]:
# usage
save_sequences_to_fasta(hsg_3utr_seq, "hsg_3utr_sequences2.fasta")

SEPTIN7 434
AATBC 434
ABCB10 434
ABCC12 434
ABCC6 434
ABCD4 434
ABHD17A 434
ACTR3B 434
ADARB1 434
ADORA2A-AS1 434
AFF3 434
AGR3 434
AHRR 434
ALOX5 434
AMY1A 434
AMY1B 434
AMY2A 434
AMY2B 434
ANAPC1 434
ANKRD30A 434
ANXA8L1 434
APEH 434
APOL1 434
APPBP2 434
AQP7 434
AR 434
AREG 434
ARHGAP11B 434
ARHGAP15 434
ARHGAP42 434
ASPM 434
ATP9B 434
BAZ2A 434
BMPR1A 434
BOLA2 434
BOLL 434
BTNL2 434
C20orf203 434
C2orf78 434
C3orf36 434
CAPN1 434
CASP12 434
CATSPER2 434
CCDC127 434
CCDC74B 434
CCKAR 434
CCT8L2 434
CCZ1B 434
CD24 434
CD44 434
CDH12 434
CDK5RAP2 434
CELSR2 434
CEP170 434
CFAP99 434
CFC1 434
CHRFAM7A 434
CHRM3 434
CLEC10A 434
CLLU1 434
CLTCL1 434
CNTNAP2 434
COMMD4 434
COPS7B 434
CRHR1 434
CROCC 434
CROCCP2 434
CSH2 434
CSPG4 434
CT45A1 434
CTAGE15 434
DDR2 434
DDX11 434
DDX54 434
DEFB103B 434
DEFB105A 434
DEFB106A 434
DGCR6L 434
DHRS4L2 434
DIXDC1 434
DMRTC1B 434
DNAH10OS 434
DPY19L2 434
DRD5 434
DSG3 434
DUSP7 434
DYNC1I2 434
E2F6 434
ECHDC3 434
EIF1AY 434
EIF3A 434
EIF3CL 434
ELN 

## upload hsg_duplicate results for easier viewing

In [ ]:
## do the filtering from mrna meatches to the match information here ..
## figure out about non-coding matches

In [16]:
mrna_matches=pd.read_csv("/scratch200/reutj/data/blast_search/mrna_matches.txt",sep="\t",
                        names=['match_chr', 'match_start', 'match_end','query_id','match_chr2','ensembl','match_type','region_start','region_end','.',
                                  'strand','phase','gene_attributes'])

In [14]:
paralog_matches=pd.read_csv("/scratch200/reutj/data/blast_search/filtered_paralog_results.txt", sep="\t",
                           names=["query_id","match_chr","identity","match_length","mismatches","gap_openings","query_start",
                                  "query_end","match_start","match_end","e_value","bit_score"],)

In [ ]:
#differences between the 2 databases
#we would like to filter on query_id, "chr", "start" , "end" (all of the last three belong to the subject)

In [102]:
mrna_matches.loc[mrna_matches["query_id"]=="ENST00000579737",:]

,match_chr,match_start,match_end,query_id,match_chr2,ensembl,match_type,region_start,region_end,.,strand,phase,gene_attributes
3316,1,149791118,149791517,ENST00000579737,1,havana,CDS,149791237,149791390,.,+,2,"gene_id ""ENSG00000150337""; gene_version ""14""; ..."
3317,1,149791118,149791517,ENST00000579737,1,ensembl_havana,CDS,149791237,149791514,.,+,2,"gene_id ""ENSG00000150337""; gene_version ""14""; ..."


In [15]:
paralog_matches.loc[paralog_matches["query_id"]=="ENST00000579737",:]

,query_id,match_chr,identity,match_length,mismatches,gap_openings,query_start,query_end,match_start,match_end,e_value,bit_score
28710,ENST00000579737,chr1,100.00,400,0,0,1,400,143883176,143883575,0.0,739.0
28711,ENST00000579737,chr1,99.75,400,1,0,1,400,121095753,121096152,0.0,734.0
28712,ENST00000579737,chr1,99.50,400,2,0,1,400,149791118,149791517,0.0,728.0


In [16]:
#remove "chr" from chr column in paralog matches
paralog_matches["match_chr"]=[i.replace("chr","") for i in paralog_matches["match_chr"]]

In [19]:
merged_mrna_matches=pd.merge(mrna_matches,paralog_matches, how="inner", on=["query_id", "match_chr","match_start","match_end"])

In [88]:
merged_mrna_matches.columns

Index(['match_chr', 'match_start', 'match_end', 'query_id', 'match_chr2',
       'ensembl', 'match_type', 'region_start', 'region_end', '.', 'strand',
       'phase', 'gene_attributes', 'identity', 'match_length', 'mismatches',
       'gap_openings', 'query_start', 'query_end', 'e_value', 'bit_score'],
      dtype='object')

In [ ]:
#do the same on the gene

In [17]:
gene_matches=pd.read_csv("/scratch200/reutj/data/blast_search/gene_matches.txt",sep="\t",
                        names=['match_chr', 'match_start', 'match_end','query_id','match_chr2','ensembl','match_type','region_start','region_end','.',
                                  'strand','phase','gene_attributes'])

In [18]:
merged_gene_matches=pd.merge(gene_matches,paralog_matches, how="inner", on=["query_id", "match_chr","match_start","match_end"])

In [19]:
len(merged_gene_matches["query_id"].unique())

212

In [25]:
len(merged_mrna_matches["query_id"].unique())

276

In [23]:
merged_mrna_matches["match_gene"]=[i.split("\"")[11] if "havana" not in i.split("\"")[11] else "" for i in merged_mrna_matches["gene_attributes"]]
merged_mrna_matches["match_ens"]=[i.split("\"")[1] for i in merged_mrna_matches["gene_attributes"]]
merged_mrna_matches["match_transcript"]=[i.split("\"")[5] for i in merged_mrna_matches["gene_attributes"]]

In [93]:
merged_mrna_matches

,match_chr,match_start,match_end,query_id,match_chr2,ensembl,match_type,region_start,region_end,.,...,match_length,mismatches,gap_openings,query_start,query_end,e_value,bit_score,match_gene,match_ens,match_transcript
0,X,153743335,153743730,ENST00000445663,X,ensembl_havana,CDS,153743221,153743346,.,...,399,15,2,3,400,5.410000e-178,628.0,ABCD1,ENSG00000101986,ENST00000218104
1,X,153743335,153743730,ENST00000445663,X,ensembl_havana,CDS,153743489,153743732,.,...,399,15,2,3,400,5.410000e-178,628.0,ABCD1,ENSG00000101986,ENST00000218104
2,10,45445718,45446117,ENST00000374391,10,ensembl_havana,three_prime_utr,45445688,45446117,.,...,400,0,0,1,400,0.000000e+00,739.0,,ENSG00000012779,ENST00000374391
3,10,45445718,45446117,ENST00000374391,10,ensembl,three_prime_utr,45445688,45446119,.,...,400,0,0,1,400,0.000000e+00,739.0,,ENSG00000012779,ENST00000542434
4,1,103664155,103664554,ENST00000370083,1,ensembl_havana,CDS,103664331,103664517,.,...,400,0,0,1,400,0.000000e+00,739.0,AMY1A,ENSG00000237763,ENST00000370083
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1688,2,106510696,106511095,ENST00000643224,2,ensembl_havana,three_prime_utr,106506941,106511095,.,...,400,0,0,1,400,0.000000e+00,739.0,,ENSG00000254126,ENST00000643224
1689,1,149470956,149471355,ENST00000593495,1,havana,three_prime_utr,149464154,149471833,.,...,400,4,0,1,400,0.000000e+00,717.0,,ENSG00000286219,ENST00000650865
1690,1,149471434,149471833,ENST00000650865,1,havana,three_prime_utr,149464154,149471833,.,...,400,0,0,1,400,0.000000e+00,739.0,,ENSG00000286219,ENST00000650865
1691,1,149791118,149791517,ENST00000579737,1,havana,CDS,149791237,149791390,.,...,400,2,0,1,400,0.000000e+00,728.0,FCGR1A,ENSG00000150337,ENST00000444948


In [20]:
merged_gene_matches["match_gene"]=[i.split("\"")[5] if "havana" not in i.split("\"")[5] else "" for i in merged_gene_matches["gene_attributes"]]
merged_gene_matches["match_ens"]=[i.split("\"")[1] for i in merged_gene_matches["gene_attributes"]]

In [21]:
merged_gene_matches

,match_chr,match_start,match_end,query_id,match_chr2,ensembl,match_type,region_start,region_end,.,...,identity,match_length,mismatches,gap_openings,query_start,query_end,e_value,bit_score,match_gene,match_ens
0,17,51225459,51225492,ENST00000400385,17,ensembl_havana,gene,51177425,51260163,.,...,97.059,34,1,0,268,301,3.200000e-06,58.4,MBTD1,ENSG00000011258
1,4,153564434,153564470,ENST00000311303,4,ensembl_havana,gene,153466346,153636711,.,...,100.000,37,0,0,361,397,1.480000e-09,69.4,TMEM131L,ENSG00000121210
2,18,51405501,51405540,ENST00000311303,18,havana,gene,51346249,51643939,.,...,95.000,40,2,0,357,396,6.890000e-08,63.9,LINC01630,ENSG00000227115
3,X,40278616,40278657,ENST00000311303,X,havana,gene,40262917,40299061,.,...,97.619,42,1,0,355,396,1.140000e-10,73.1,,ENSG00000236393
4,1,148149557,148149788,ENST00000292577,1,ensembl_havana,gene,148146395,148149566,.,...,99.138,232,1,1,1,231,4.590000e-114,416.0,ABHD17AP1,ENSG00000198658
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10240,18,54926267,54926297,ENST00000664044,18,ensembl_havana,gene,54901509,54959461,.,...,100.000,31,0,0,54,84,3.200000e-06,58.4,CCDC68,ENSG00000166510
10241,4,49230738,49230768,ENST00000664044,4,havana,gene,49229573,49231284,.,...,100.000,31,0,0,2,32,3.200000e-06,58.4,,ENSG00000280043
10242,13,99510434,99510467,ENST00000664044,13,ensembl_havana,gene,99446311,99564048,.,...,97.059,34,1,0,1,34,3.200000e-06,58.4,TM9SF2,ENSG00000125304
10243,22,25906709,25906751,ENST00000664044,22,ensembl_havana,gene,25742144,26031045,.,...,95.349,43,2,0,52,94,1.480000e-09,69.4,MYO18B,ENSG00000133454


In [ ]:
#gene is much better lets keep it

In [22]:
#lets add gene names to the query
query_gene=[]
for i in merged_gene_matches["query_id"]:
    for k,v in hsg_3utr_seq.items():
        if i in v:
            query_gene.append(k)
merged_gene_matches["query_gene"]=query_gene

In [23]:
query_gene=[]
for i in merged_mrna_matches["query_id"]:
    for k,v in hsg_3utr_seq.items():
        if i in v:
            query_gene.append(k)
merged_mrna_matches["query_gene"]=query_gene

NameError: name 'merged_mrna_matches' is not defined

In [37]:
query_ens=[]
for i in merged_mrna_matches["query_gene"]:
    count=0
    for j in range(len(hsg_ens)):
        if i==hsg_symbols[j]:
            count+=1
            if count<2:#add only the first ens that matches, individual cases can be delt with later
                query_ens.append(hsg_ens[j])
merged_mrna_matches["query_ens"]=query_ens

In [27]:
hsg_ens=hsg["ens"].tolist()

In [28]:
query_ens=[]
for i in merged_gene_matches["query_gene"]:
    count=0
    for j in range(len(hsg_ens)):
        if i==hsg_symbols[j]:
            count+=1
            if count<2:#add only the first ens that matches, individual cases can be delt with later
                query_ens.append(hsg_ens[j])
merged_gene_matches["query_ens"]=query_ens

In [25]:
#drop and rearrange
#merged_mrna_matches=merged_mrna_matches.drop(columns=['match_chr2','ensembl','.', 'strand','phase', 'gene_attributes'])
merged_gene_matches=merged_gene_matches.drop(columns=['match_chr2','ensembl','.','match_type', 'strand','phase', 'gene_attributes'])

In [40]:
merged_mrna_matches=merged_mrna_matches.loc[:,['query_gene','query_ens','query_id','match_gene', 'match_ens', 'match_transcript','match_type','match_chr', 'match_start', 'match_end','identity', 'match_length','mismatches','gap_openings',
                                               'e_value','bit_score','query_start','query_end']]

In [29]:
merged_gene_matches=merged_gene_matches.loc[:,['query_gene','query_ens','query_id','match_gene', 'match_ens', 
                                     'match_chr', 'match_start', 'match_end','identity', 'match_length','mismatches','gap_openings',
                                               'e_value','bit_score','query_start','query_end']]

In [41]:
merged_mrna_matches

,query_gene,query_ens,query_id,match_gene,match_ens,match_transcript,match_type,match_chr,match_start,match_end,identity,match_length,mismatches,gap_openings,e_value,bit_score,query_start,query_end
0,ABCD1P5,ENSG00000214330,ENST00000445663,ABCD1,ENSG00000101986,ENST00000218104,CDS,X,153743335,153743730,95.238,399,15,2,5.410000e-178,628.0,3,400
1,ABCD1P5,ENSG00000214330,ENST00000445663,ABCD1,ENSG00000101986,ENST00000218104,CDS,X,153743335,153743730,95.238,399,15,2,5.410000e-178,628.0,3,400
2,ALOX5,ENSG00000012779,ENST00000374391,,ENSG00000012779,ENST00000374391,three_prime_utr,10,45445718,45446117,100.000,400,0,0,0.000000e+00,739.0,1,400
3,ALOX5,ENSG00000012779,ENST00000374391,,ENSG00000012779,ENST00000542434,three_prime_utr,10,45445718,45446117,100.000,400,0,0,0.000000e+00,739.0,1,400
4,AMY1A,ENSG00000237763,ENST00000370083,AMY1A,ENSG00000237763,ENST00000370083,CDS,1,103664155,103664554,100.000,400,0,0,0.000000e+00,739.0,1,400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1688,CD8B2,ENSG00000254126,ENST00000643224,,ENSG00000254126,ENST00000643224,three_prime_utr,2,106510696,106511095,100.000,400,0,0,0.000000e+00,739.0,1,400
1689,NOTCH2NLB,ENSG00000286019,ENST00000593495,,ENSG00000286219,ENST00000650865,three_prime_utr,1,149470956,149471355,99.000,400,4,0,0.000000e+00,717.0,1,400
1690,NOTCH2NLC,ENSG00000286219,ENST00000650865,,ENSG00000286219,ENST00000650865,three_prime_utr,1,149471434,149471833,100.000,400,0,0,0.000000e+00,739.0,1,400
1691,FCGR1CP,ENSG00000265531,ENST00000579737,FCGR1A,ENSG00000150337,ENST00000444948,CDS,1,149791118,149791517,99.500,400,2,0,0.000000e+00,728.0,1,400


In [30]:
merged_gene_matches

,query_gene,query_ens,query_id,match_gene,match_ens,match_chr,match_start,match_end,identity,match_length,mismatches,gap_openings,e_value,bit_score,query_start,query_end
0,AATBC,ENSG00000215458,ENST00000400385,MBTD1,ENSG00000011258,17,51225459,51225492,97.059,34,1,0,3.200000e-06,58.4,268,301
1,ABCC12,ENSG00000140798,ENST00000311303,TMEM131L,ENSG00000121210,4,153564434,153564470,100.000,37,0,0,1.480000e-09,69.4,361,397
2,ABCC12,ENSG00000140798,ENST00000311303,LINC01630,ENSG00000227115,18,51405501,51405540,95.000,40,2,0,6.890000e-08,63.9,357,396
3,ABCC12,ENSG00000140798,ENST00000311303,,ENSG00000236393,X,40278616,40278657,97.619,42,1,0,1.140000e-10,73.1,355,396
4,ABHD17A,ENSG00000129968,ENST00000292577,ABHD17AP1,ENSG00000198658,1,148149557,148149788,99.138,232,1,1,4.590000e-114,416.0,1,231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10240,LINC00868,ENSG00000267535,ENST00000664044,CCDC68,ENSG00000166510,18,54926267,54926297,100.000,31,0,0,3.200000e-06,58.4,54,84
10241,LINC00868,ENSG00000267535,ENST00000664044,,ENSG00000280043,4,49230738,49230768,100.000,31,0,0,3.200000e-06,58.4,2,32
10242,LINC00868,ENSG00000267535,ENST00000664044,TM9SF2,ENSG00000125304,13,99510434,99510467,97.059,34,1,0,3.200000e-06,58.4,1,34
10243,LINC00868,ENSG00000267535,ENST00000664044,MYO18B,ENSG00000133454,22,25906709,25906751,95.349,43,2,0,1.480000e-09,69.4,52,94


In [ ]:
#genes_w_match=list(merged_gene_matches["query_gene"].unique())
#no_95_similarity=[i for i in hsg_3utr_seq.keys() if i not in genes_w_match]

In [31]:
#paralogs- ones with different ensg and/or gene names . for the first step lets filter
#paralogs_coding=merged_mrna_matches.query("query_ens!=match_ens")
paralogs_genes=merged_gene_matches.query("query_ens!=match_ens")

In [32]:
#filter for match length> 300 and max 4 mismatches
paralogs_genes= paralogs_genes.loc[(paralogs_genes["match_length"]>=300) & (paralogs_genes["mismatches"]<=4) ,:]
#paralogs_coding= paralogs_coding.loc[(paralogs_coding["match_length"]>=300) & (paralogs_coding["mismatches"]<=4) ,:]

In [33]:
#remove duplicates with the exact match location
#paralogs_coding=paralogs_coding.drop_duplicates(subset=['query_ens','query_id', 'match_ens', 'match_chr','match_start','match_end','match_type'])
paralogs_genes=paralogs_genes.drop_duplicates(subset=['query_ens','query_id', 'match_ens', 'match_chr','match_start','match_end'])

In [35]:
#paralogs_coding.query("query_gene==match_gene")
#check if they have any duplicate ens and remove all of them if gene name is the same or location of gene
#(by other line with the same gene name) and no gene name matched to it , also fusion genes with the same gene

In [68]:
paralogs_coding.loc[paralogs_coding["query_gene"]=="PRAMEF25",:]

,query_gene,query_ens,query_id,match_gene,match_ens,match_transcript,match_type,match_chr,match_start,match_end,identity,match_length,mismatches,gap_openings,e_value,bit_score,query_start,query_end
1103,PRAMEF25,ENSG00000204505,ENST00000619661,PRAMEF25,ENSG00000229571,ENST00000614839,CDS,1,13077485,13077884,100.0,400,0,0,0.0,739.0,1,400
1104,PRAMEF25,ENSG00000204505,ENST00000619661,,ENSG00000229571,ENST00000614839,three_prime_utr,1,13077485,13077884,100.0,400,0,0,0.0,739.0,1,400


In [69]:
paralogs_coding=paralogs_coding.drop([993,994, 1103,1104,1621])

In [36]:
paralogs_genes.query("query_gene==match_gene")
#check if they have any duplicate ens and remove all of them including transcripts with the same locations..

,query_gene,query_ens,query_id,match_gene,match_ens,match_chr,match_start,match_end,identity,match_length,mismatches,gap_openings,e_value,bit_score,query_start,query_end
24,ARHGAP11B,ENSG00000187951,ENST00000697964,ARHGAP11B,ENSG00000285077,15,30638411,30638810,100.0,400,0,0,0.0,739.0,1,400
161,F8A2,ENSG00000274791,ENST00000369505,F8A2,ENSG00000288709,X,155383402,155383801,100.0,400,0,0,0.0,739.0,1,400
285,FAM95B1,ENSG00000223839,ENST00000661710,FAM95B1,ENSG00000290718,9,40326047,40326446,100.0,400,0,0,0.0,739.0,1,400
1004,IL9R,ENSG00000124334,ENST00000711286,IL9R,ENSG00000292373,Y,57196938,57197337,100.0,400,0,0,0.0,739.0,1,400
7848,OR4M2,ENSG00000182974,ENST00000614722,OR4M2,ENSG00000274102,15,22081211,22081610,100.0,400,0,0,0.0,739.0,1,400
9798,VAMP7,ENSG00000124333,ENST00000711260,VAMP7,ENSG00000292366,Y,57129890,57130289,100.0,400,0,0,0.0,739.0,1,400
9931,FRMPD2B,ENSG00000293529,ENST00000431305,FRMPD2B,ENSG00000150175,10,46894163,46894562,100.0,400,0,0,0.0,739.0,1,400
9939,GTF2IP4,ENSG00000293241,ENST00000453092,GTF2IP4,ENSG00000233369,7,73205042,73205441,100.0,400,0,0,0.0,739.0,1,400
9964,NCF1B,ENSG00000290838,ENST00000435988,NCF1B,ENSG00000182487,7,73235391,73235790,100.0,400,0,0,0.0,739.0,1,400
9985,ROCK1P1,ENSG00000263006,ENST00000573767,ROCK1P1,ENSG00000290877,18,118105,118504,100.0,400,0,0,0.0,739.0,1,400


In [46]:
paralogs_genes.loc[paralogs_genes["query_gene"]=="ROCK1P1",:]

,query_gene,query_ens,query_id,match_gene,match_ens,match_chr,match_start,match_end,identity,match_length,mismatches,gap_openings,e_value,bit_score,query_start,query_end
9985,ROCK1P1,ENSG00000263006,ENST00000573767,ROCK1P1,ENSG00000290877,18,118105,118504,100.0,400,0,0,0.0,739.0,1,400


In [47]:
paralogs_genes=paralogs_genes.drop([24,25,161,285,286,1003,1004,7848,7849,9798,9931,9939,9964,9985])

In [94]:
paralogs_coding

,query_gene,query_ens,query_id,match_gene,match_ens,match_transcript,match_type,match_chr,match_start,match_end,identity,match_length,mismatches,gap_openings,e_value,bit_score,query_start,query_end
6,AMY1A,ENSG00000237763,ENST00000370083,AMY1C,ENSG00000187733,ENST00000684141,CDS,1,103758293,103758692,100.000,400,0,0,0.0,739.0,1,400
7,AMY1A,ENSG00000237763,ENST00000370083,,ENSG00000187733,ENST00000684141,three_prime_utr,1,103758293,103758692,100.000,400,0,0,0.0,739.0,1,400
17,AMY1B,ENSG00000174876,ENST00000330330,AMY1A,ENSG00000237763,ENST00000370083,CDS,1,103664155,103664554,100.000,400,0,0,0.0,739.0,1,400
18,AMY1B,ENSG00000174876,ENST00000330330,,ENSG00000237763,ENST00000370083,three_prime_utr,1,103664155,103664554,100.000,400,0,0,0.0,739.0,1,400
19,AMY1B,ENSG00000174876,ENST00000330330,AMY1C,ENSG00000187733,ENST00000684141,CDS,1,103758293,103758692,100.000,400,0,0,0.0,739.0,1,400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1670,PTPN20CP,ENSG00000278561,ENST00000614090,,ENSG00000204179,ENST00000510335,three_prime_utr,10,46987154,46987553,100.000,400,0,0,0.0,739.0,1,400
1675,GTF2IP4,ENSG00000293241,ENST00000453092,GTF2I,ENSG00000263001,ENST00000650807,CDS,7,74758438,74758838,99.501,401,1,1,0.0,728.0,1,400
1686,GPR89B,ENSG00000188092,ENST00000314163,,ENSG00000117262,ENST00000460277,three_prime_utr,1,145670316,145670719,98.020,404,4,1,0.0,699.0,1,400
1689,NOTCH2NLB,ENSG00000286019,ENST00000593495,,ENSG00000286219,ENST00000650865,three_prime_utr,1,149470956,149471355,99.000,400,4,0,0.0,717.0,1,400


In [48]:
paralogs_genes

,query_gene,query_ens,query_id,match_gene,match_ens,match_chr,match_start,match_end,identity,match_length,mismatches,gap_openings,e_value,bit_score,query_start,query_end
8,ALOX5,ENSG00000012779,ENST00000374391,,ENSG00000231964,10,45445718,45446117,100.0,400,0,0,0.0,739.0,1,400
11,AMY1A,ENSG00000237763,ENST00000370083,AMY1C,ENSG00000187733,1,103758293,103758692,100.0,400,0,0,0.0,739.0,1,400
13,AMY1B,ENSG00000174876,ENST00000330330,AMY1A,ENSG00000237763,1,103664155,103664554,100.0,400,0,0,0.0,739.0,1,400
14,AMY1B,ENSG00000174876,ENST00000330330,AMY1C,ENSG00000187733,1,103758293,103758692,100.0,400,0,0,0.0,739.0,1,400
17,AMY2A,ENSG00000243480,ENST00000414303,AMYP1,ENSG00000227408,1,103719506,103719905,100.0,400,0,0,0.0,739.0,1,400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9993,NPY4R,ENSG00000204174,ENST00000374312,NPY4R2,ENSG00000264717,10,47923125,47923524,100.0,400,0,0,0.0,739.0,1,400
9994,NPY4R,ENSG00000204174,ENST00000374312,,ENSG00000276850,10,47923125,47923524,100.0,400,0,0,0.0,739.0,1,400
9996,OR2A1,ENSG00000221970,ENST00000641044,,ENSG00000284644,7,144322338,144322737,100.0,400,0,0,0.0,739.0,1,400
9997,OR2A1,ENSG00000221970,ENST00000641044,OR2A1-AS1,ENSG00000244479,7,144322338,144322737,100.0,400,0,0,0.0,739.0,1,400


In [123]:
paralogs_genes.index

Index([   14,    19,    22,    24,    25,    28,    32,    33,    34,    35,
       ...
       11775, 11777, 11780, 11781, 11782, 11783, 11784, 11785, 11786, 11787],
      dtype='int64', length=443)

In [49]:
#remove fusion genes
fusion=list(set([i for i in paralogs_genes["match_gene"] if "-" in i]))
paralogs_genes=paralogs_genes.loc[~paralogs_genes["match_gene"].isin(fusion),:]

In [129]:
#remove fusion genes-coding
fusion=list(set([i for i in paralogs_coding["match_gene"] if "-" in i]))
paralogs_coding=paralogs_coding.loc[~paralogs_coding["match_gene"].isin(fusion),:]

In [50]:
#how much unique genes we have total
print(len(list(paralogs_genes["query_ens"].unique())))
#print(len(list(paralogs_coding["query_ens"].unique())))

102


In [51]:
paralogs_genes["query_gene"].unique()

array(['ALOX5', 'AMY1A', 'AMY1B', 'AMY2A', 'ANXA8L1', 'ARHGAP11B',
       'CCZ1B', 'CFC1', 'CHRFAM7A', 'CROCCP2', 'DEFB103B', 'DGCR6L',
       'DHRS4L2', 'DRD5', 'EIF3CL', 'F8A2', 'F8A3', 'FAM153A', 'FAM156A',
       'FAM25G', 'FAM3C', 'FAM72C', 'FAM95B1', 'FCGR2C', 'FGF7',
       'GOLGA8G', 'GPR42', 'GTF2H2C', 'IL9R', 'LRRC37A', 'MICA',
       'NANOGP8', 'NBPF12', 'NBPF15', 'NOMO3', 'NUTM2A', 'NUTM2B',
       'NUTM2D', 'OR11H1', 'OR2A42', 'OR4F16', 'OR4F17', 'OR4F21',
       'OR4F3', 'OR4F4', 'OR4F5', 'OR4M1', 'OR4M2', 'OR4N4', 'OR4Q3',
       'PAIP1', 'PARP4', 'PCDHB13', 'PDCD4', 'POLR2J2', 'POM121',
       'RAB3IP', 'RFPL4A', 'RGPD2', 'RGPD3', 'RGPD6', 'RGPD8', 'RIMBP3B',
       'RIMBP3C', 'SDHA', 'SERF1B', 'SLC29A4', 'SLX1B', 'SMN2', 'SPACA5B',
       'SPANXA1', 'SPATA31A1', 'SPATA31A3', 'SPATA31A5', 'SPATA31A7',
       'SPDYE1', 'SRGAP2C', 'STAG3', 'TBC1D3', 'TCAF2', 'THOC3',
       'TP53TG3', 'TRIM74', 'ZNF492', 'ARHGEF34P', 'ARHGEF35', 'FAM72B',
       'FCGR1CP', 'GPR89B', 'GTF2

In [ ]:
#genes_w_paralog=list(paralogs["queri_gene"].unique())
#gene_wo_parlog=[i for i in genes_w_match if i not in genes_w_paralog] 

In [ ]:
#upload to go over again

In [14]:
hsg_3utr_seq_df=pd.read_csv("/scratch200/reutj/data/hsg_3utr_seq_df.csv")
hsg_df=pd.read_csv("/scratch200/reutj/data/hsg_genes.csv")

In [15]:
hsg_ens=list(hsg_df["ens"])
hsg_symbols=list(hsg_df["gene"])

In [32]:
hsg_3utr_seq=dict(zip(hsg_3utr_seq_df.gene, hsg_3utr_seq_df.seq))

In [2]:
paralogs_genes=pd.read_csv("/scratch200/reutj/data/hsg_with_paralogs.csv")
paralogs_coding=pd.read_csv("/scratch200/reutj/data/hsg_with_coding_paralogs.csv")

In [52]:
paralogs_genes.to_csv("/scratch200/reutj/data/hsg_with_paralogs.csv")
#paralogs_coding.to_csv("/scratch200/reutj/data/hsg_with_coding_paralogs.csv")